# 02 · PCA 분해 & 온도 의존성

**워크플로**
1. 01에서 저장한 온도별 `npz`(G(r))를 모두 불러오기
2. G(r)를 **무지개색 워터폴**로 쌓아 보기 (겹치지 않게 offset)
3. **PCA**로 분해 → 평균 G(r) + 변동 모드(PC1·PC2). (G(r)은 음수가 있어 NMF 부적합)
4. **PC score** vs 온도
5. **1st peak 위치 변화** — 가우시안 피팅
5b. **차분맵 ΔG(r,T)** — ripple 상쇄, 변화만
6. **2nd shell·FSDP** — 중거리 질서 트렌드

재사용 함수는 모두 `fourdstem`에 있습니다:
`decompose_profiles`, `plot_series_waterfall`, `fit_gaussian_peak`, `track_peak`.

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
import fourdstem as fds

OUT_DIR = "/home/jonghoonk918/Desktop/fdstem/Amorphous/In-situ/Heating-SiOrdf_npz"  # 01 노트북의 저장 폴더
if not os.path.isdir(OUT_DIR):
    OUT_DIR = "rdf_npz"            # 데모/검증 실행용 로컬 폴더
N_COMPONENTS = 2                    # PCA 성분(주성분) 수
FIRST_PEAK_WINDOW = (1.45, 1.85)   # 1st peak 가우시안 피팅 구간 (Å) — 온도 이동 여유

## 1) NPZ 불러오기 (온도별 G(r))

온도 순으로 정렬하고, 공통 r 축으로 맞춰 행렬 `X (n_temperature × n_r)`를 만듭니다.

In [ ]:
files = sorted(glob.glob(os.path.join(OUT_DIR, "*_rdf.npz")))
assert files, f"{OUT_DIR}/ 에 npz가 없습니다. 먼저 01 노트북을 실행하세요."

records = []
for p in files:
    d = fds.load_result_npz(p)
    records.append((float(d["temperature"]), np.asarray(d["r"]), np.asarray(d["Gr"]),
                    np.asarray(d["q_reduced"]), np.asarray(d["phi"])))
records.sort(key=lambda t: t[0])

temps = np.array([t[0] for t in records])
r_ref = records[0][1]
# 공통 r축으로 보간(그리드가 다를 수 있으므로)
X = np.vstack([np.interp(r_ref, r, Gr) for _, r, Gr, _, _ in records])
profiles = [(r_ref, X[i]) for i in range(len(temps))]            # (r, G(r)) per T
phi_profiles = [(rec[3], rec[4]) for rec in records]             # (q, φ(q)) per T
print("temperatures:", temps)
print("X shape (n_T × n_r):", X.shape)

## 2) 온도별 G(r) 무지개 워터폴

`plot_series_waterfall`은 온도에 따라 **무지개색**으로 칠하고, `offset`만큼 위로 쌓아 겹치지 않게 합니다.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))
fds.plot_series_waterfall(profiles, temps, ax=ax, cmap="rainbow",
                          xlabel="r (Å)", labels=[f"{int(T)}K" for T in temps])
ax.set_title("G(r) per temperature (rainbow waterfall)")
plt.show()

## 3) PCA 분해 — G(r) 변동 모드 (NMF 아님!)

**주의**: G(r)은 **음수(골짜기·baseline)** 가 있어 **NMF(비음수 전용)로 분해하면 안 됩니다** — 음수가 0으로
잘려 성분이 왜곡됩니다(RDF처럼 안 보이는 이유). 부호 있는 G(r) 시계열엔 **PCA**가 정답:

`X = 평균 + Σ score_i · PC_i`
- **평균 G(r)** = 모든 온도의 공통 구조
- **PC1, PC2** = 온도에 따라 G(r)가 **어떻게 변하는지의 모드**(음수 포함, RDF 차분처럼 생김)
- **explained variance** = 각 모드가 설명하는 변동 비율

(NMF는 회절 패턴·I(q) 같은 **비음수** 데이터 전용입니다.)

In [ ]:
dec = fds.decompose_profiles(X, n_components=N_COMPONENTS, x=r_ref, method="pca")
evr = dec.explained_variance_ratio
print("explained variance ratio:", np.round(evr, 4))
mean_g = dec.model.mean_

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(r_ref, mean_g, "k", lw=1.5); ax[0].axhline(0, color="0.8", lw=0.8)
ax[0].set_xlabel("r (Å)"); ax[0].set_ylabel("G(r)"); ax[0].set_title("mean G(r) (common structure)")
for j in range(N_COMPONENTS):
    ax[1].plot(r_ref, dec.components[j], lw=1.5,
               label=f"PC{j+1} ({evr[j]*100:.1f}%)")
ax[1].axhline(0, color="0.8", lw=0.8)
ax[1].set_xlabel("r (Å)"); ax[1].set_ylabel("PC (mode of variation)")
ax[1].set_title("PCA components — how G(r) changes with T"); ax[1].legend()
plt.tight_layout(); plt.show()

## 4) PC score vs 온도 — 변동의 크기·방향

각 온도에서 **PC1/PC2 score**(= 그 모드가 얼마나 섞였나, 부호 가능). PC1 score의 온도 변화가 **주된 구조
변화 추세**입니다. 부드러운 단조 변화면 점진적 재배열, 급변점이 있으면 전이/결정화 신호.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for j in range(N_COMPONENTS):
    ax.plot(temps, dec.weights[:, j], "o-", label=f"PC{j+1} ({evr[j]*100:.1f}%)")
ax.axhline(0, color="0.8", lw=0.8)
ax.set_xlabel("temperature (K)"); ax.set_ylabel("PC score")
ax.set_title("PCA score vs temperature"); ax.legend()
plt.show()

## 5) 1st peak 위치 변화 — 가우시안 피팅 (~1.6 Å)

각 온도의 G(r)에서 첫 배위 피크를 **가우시안으로 피팅**해 중심을 추출합니다.
> 참고: **Si–O 최근접 결합(~1.6 Å)은 온도에 거의 불변**입니다(강직한 SiO₄ 사면체). 첫 피크가 거의
> 평평하게 나오면 정상이며, 온도 신호는 아래 **6) 2nd shell·FSDP**나 PCA score에서 더 잘 보입니다.

In [ ]:
centers, sigmas = [], []
for (r, Gr) in profiles:
    fit = fds.fit_gaussian_peak(r, Gr, *FIRST_PEAK_WINDOW)
    centers.append(fit["center"]); sigmas.append(fit["sigma"])
centers = np.array(centers); sigmas = np.array(sigmas)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
# (좌) 대표 온도들의 피팅 겹쳐 보기
cm = plt.get_cmap("rainbow")
for i in range(0, len(temps), max(1, len(temps)//5)):
    r, Gr = profiles[i]
    sel = (r >= FIRST_PEAK_WINDOW[0]-0.15) & (r <= FIRST_PEAK_WINDOW[1]+0.15)
    c = cm(i/max(len(temps)-1, 1))
    ax[0].plot(r[sel], Gr[sel], ".", color=c, ms=3)
    fit = fds.fit_gaussian_peak(r, Gr, *FIRST_PEAK_WINDOW)
    ax[0].plot(fit["xfit"], fit["yfit"], "-", color=c, lw=1.5,
               label=f"{int(temps[i])}K")
ax[0].axvspan(*FIRST_PEAK_WINDOW, color="0.9", zorder=0)
ax[0].set_xlabel("r (Å)"); ax[0].set_ylabel("G(r)")
ax[0].set_title("1st peak Gaussian fit"); ax[0].legend(fontsize=8)

# (우) 첫 피크 중심 vs 온도
ax[1].errorbar(temps, centers, yerr=sigmas, fmt="o-", color="crimson", capsize=3)
ax[1].set_xlabel("temperature (K)"); ax[1].set_ylabel("1st peak center (Å)")
ax[1].set_title("1st-neighbour distance vs T")
plt.tight_layout(); plt.show()

for T, c, s in zip(temps, centers, sigmas):
    print(f"  T={T:>5.0f}K   center={c:.3f} Å   σ={s:.3f} Å")

## 5b) 차분 ΔG(r,T) — ripple 상쇄, 변화만 또렷하게

모든 온도가 **같은 q 구간**으로 변환되므로 **termination ripple은 모든 프레임에서 동일**합니다.
따라서 **기준(최저온)을 빼면 ripple은 상쇄되고 진짜 온도 변화만 남습니다** — 신호를 "확 보이게" 하는
올바른 방법(q_min을 키우는 게 아니라). 워터폴 + 2D 히트맵으로 봅니다.

In [ ]:
G0 = X[0]                                  # 기준 = 최저온
dG = X - G0[None, :]                        # ΔG(r,T) = G(r,T) - G(r, T_ref)

fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
# (좌) ΔG 워터폴
fds.plot_series_waterfall([(r_ref, dG[i]) for i in range(len(temps))], temps,
                          ax=ax[0], cmap="rainbow", xlabel="r (Å)")
ax[0].set_title(f"ΔG(r) = G(r,T) - G(r,{int(temps[0])}K)   (ripple cancels)")
# (우) 2D 히트맵 (온도 × r)
vmax = np.nanpercentile(np.abs(dG), 99) or 1.0
im = ax[1].imshow(dG, aspect="auto", origin="lower", cmap="RdBu_r",
                  vmin=-vmax, vmax=vmax,
                  extent=[r_ref[0], r_ref[-1], 0, len(temps)-1])
ax[1].set_yticks(range(len(temps))); ax[1].set_yticklabels([f"{int(T)}K" for T in temps], fontsize=7)
ax[1].set_xlim(0, 6); ax[1].set_xlabel("r (Å)"); ax[1].set_ylabel("temperature")
ax[1].set_title("ΔG(r,T) heatmap"); fig.colorbar(im, ax=ax[1], label="ΔG", fraction=0.046)
plt.tight_layout(); plt.show()

## 6) 온도 신호가 진짜 있는 곳 — 2nd shell & FSDP

Si–O 1st shell은 거의 불변이므로, **중거리 질서 변화**를 봅니다:
- **2nd shell** (G(r) ~2.4–3.4 Å: Si···Si / O···O) 위치·세기
- **FSDP** (φ(q) 첫 피크, 중거리 질서의 지표) 위치

창(window) 값은 데이터에 맞게 조절하세요.

In [ ]:
SECOND_WINDOW = (2.3, 3.4)          # 2nd shell (Å) — 데이터 보고 조절
FSDP_WINDOW   = (0.40, 0.90)        # φ(q) 첫 피크 (1/Å, q=1/d) — 데이터 보고 조절

sec  = fds.track_peak(profiles,     temps, SECOND_WINDOW, mode="centroid")
fsdp = fds.track_peak(phi_profiles, temps, FSDP_WINDOW,   mode="max")
area = fds.integrate_region(profiles, temps, SECOND_WINDOW, baseline="linear")

fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
ax[0].plot(sec["coord"], sec["position"], "o-", color="teal")
ax[0].set_xlabel("temperature (K)"); ax[0].set_ylabel("2nd shell r (Å)")
ax[0].set_title(f"2nd shell position vs T  {SECOND_WINDOW} Å")
ax[1].plot(area["coord"], area["integral"], "s-", color="seagreen")
ax[1].set_xlabel("temperature (K)"); ax[1].set_ylabel("2nd shell area (a.u.)")
ax[1].set_title("2nd shell area vs T")
ax[2].plot(fsdp["coord"], fsdp["position"], "^-", color="darkorange")
ax[2].set_xlabel("temperature (K)"); ax[2].set_ylabel("FSDP q (1/Å)")
ax[2].set_title(f"FSDP position vs T  {FSDP_WINDOW} 1/Å")
plt.tight_layout(); plt.show()

for T, r2, a2, qf in zip(temps, sec["position"], area["integral"], fsdp["position"]):
    print(f"  T={T:>6.0f}K   2nd r={r2:.3f} Å   2nd area={a2:.3g}   FSDP q={qf:.3f} 1/Å")

**정리** — 01에서 온도별 G(r)를 만들고, 02에서 (2) 워터폴 · (3) PCA 평균/변동모드 · (4) PC score ·
(5) 1st peak · (5b) 차분맵 · (6) 2nd shell·FSDP로 온도 의존성을 정량화했습니다.
`N_COMPONENTS`, `FIRST_PEAK_WINDOW`, `SECOND_WINDOW`, `FSDP_WINDOW`만 바꿔 재사용하세요.